<a href="https://colab.research.google.com/github/karimiMahnaz/Colab_Notebooks/blob/main/TransferLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from tensorflow import keras
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D
from tensorflow.keras.utils import to_categorical
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential

In [8]:
train_path = "/content/intel-image-classification/seg_train/seg_train"
test_path = "/content/intel-image-classification/seg_test/seg_test"
pred_path = "/content/intel-image-classification/seg_pred/seg_pred"

In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [18]:
data_gen = ImageDataGenerator()
# ImageDataGenerator(rescale=1./255)

In [11]:
batch_size = 10

In [ ]:

train_data_gen = data_gen.flow_from_directory(train_path, target_size=(150,150), batch_size=32)
test_data_gen = data_gen.flow_from_directory(test_path , target_size=(150,150), batch_size=32)


In [15]:
from tensorflow.keras.applications import EfficientNetV2B1

In [16]:
model_pre = EfficientNetV2B1(include_top=False, weights='imagenet', input_shape=(150,150,3))

28456008/28456008 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [17]:
model_pre.summary()

Model: "efficientnetv2-b1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, None,      │          0 │ input_layer[0][0] │
│ (Rescaling)         │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, None,      │          0 │ rescaling[0][0]   │
│ (Normalization)     │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, None,      │        864 │ normalization[0]… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, None,      │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, None,      │          0 │ stem_bn[0][0]     │
│ (Activation)        │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, None,      │      4,608 │ stem_activation[… │
│ (Conv2D)            │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, None,      │         64 │ block1a_project_… │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, None,      │          0 │ block1a_project_… │
│ (Activation)        │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_co… │ (None, None,      │      2,304 │ block1a_project_… │
│ (Conv2D)            │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_bn  │ (None, None,      │         64 │ block1b_project_… │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_ac… │ (None, None,      │          0 │ block1b_project_… │
│ (Activation)        │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_drop        │ (None, None,      │          0 │ block1b_project_… │
│ (Dropout)           │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_add (Add)   │ (None, None,      │          0 │ block1b_drop[0][… │
│                     │ None, 16)         │            │ block1a_project_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, None,      │      9,216 │ block1b_add[0][0] │
│ (Conv2D)            │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, None,      │        256 │ block2a_expand_c… │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, None,      │          0 │ block2a_expand_b

 Total params: 6,931,124 (26.44 MB)

 Trainable params: 6,860,052 (26.17 MB)

 Non-trainable params: 71,072 (277.62 KB)

In [ ]:
model_pre.layers

In [ ]:
model_pre.layers[0].trainable = False

In [ ]:
from layers in model_pre.layers:
  print(layer)
  if layer.name == 'block_13_expand_conv':
  layer.trainable = False

In [ ]:
batch1 = train_data_gen.next()

In [ ]:
pred_pre = model_pre.predict(batch1[0])

In [ ]:
pred_pre[0]

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(batch1[0][0])

In [ ]:
# freeze the model
model_pre.trainable = False

In [ ]:
model = Sequential(name = "Pretrain_EfficientNetV2B1")
model.add(model_pre)
model.add(Flatten())
model.add(Dense(128, activation='relu', name = 'Dense_1'))
model.add(Dropout(0.5, name = 'Dropout_1'))
model.add(Dense(64, activation='relu', name = 'Dense_2'))
model.add(Dropout(0.5, name = 'Dropout_2'))
model.add(Dense(6, activation='softmax', name = 'Output'))

In [ ]:
model.summary()

In [ ]:
# complile model
opt = tf.optimizers.Adam(learning_rate=.001)
loss = tf.losses.CategoricalCrossentropy()
metric = tf.metrics.CategoricalAccuracy()
model.compile(optimizer=opt, loss=loss, metrics=[metric])

In [ ]:
model.fit(train_data_gen, epochs=10, batch_size=batch_size, validation_data=(x_val, y_val), verbose=2)